In [1]:
import pandas as pd
from pathlib import Path
import os
import plotly.express as px

os.chdir(Path.cwd().parent)

save_path= "backtest/attachments"

horizons = [1,3,5]
strategies = {"momentum_score": "Momentum Strategie", "value_score": "Value Strategie", "passive_GSPC": "Passive GSPC Strategie", "passive_SPXEW": "Passive SPXEW Strategie"}

df_strategies = pd.read_csv("backtest/data/backtest_data.csv")
df_benchmark = pd.read_csv("backtest/data/benchmark_data.csv")

In [2]:
df_strategies_grouped = df_strategies.drop(labels=["sector"], axis=1).groupby(by=["buyyear", "sellyear","strategy"], as_index=False).median()
df_strategies_grouped["horizon"] = df_strategies_grouped["sellyear"] - df_strategies_grouped["buyyear"]
df_strategies_grouped = df_strategies_grouped[df_strategies_grouped["horizon"].isin(horizons)]
df_strategies_1_year = df_strategies_grouped[df_strategies_grouped["horizon"] == 1].reset_index(drop=True)

In [3]:
df_benchmark_horizon = df_benchmark
df_benchmark_horizon["horizon"] = df_benchmark_horizon["sellyear"] - df_benchmark_horizon["buyyear"]
df_benchmark_horizon = df_benchmark_horizon[df_benchmark_horizon["horizon"].isin([1])]
df_benchmark_horizon = df_benchmark_horizon.drop(axis=1, labels=["index"]).reset_index(drop=True)

In [4]:
df_unified = pd.concat([df_benchmark_horizon, df_strategies_1_year])


pd.set_option("display.max_rows", None)
df_unified.head(70)

,buyyear,sellyear,rendite,strategy,horizon
0,2008,2009,43.260,passive_SPXEW,1
1,2009,2010,19.810,passive_SPXEW,1
2,2010,2011,-1.920,passive_SPXEW,1
3,2011,2012,15.270,passive_SPXEW,1
4,2012,2013,33.620,passive_SPXEW,1
5,2013,2014,12.350,passive_SPXEW,1
6,2014,2015,-4.110,passive_SPXEW,1
7,2015,2016,12.500,passive_SPXEW,1
8,2016,2017,16.680,passive_SPXEW,1
9,2017,2018,-9.430,passive_SPXEW,1


In [5]:
from pathlib import Path
import plotly.express as px

BLUE = "#4E79A7"        # Value
ORANGE = "#F28E2B"      # Momentum
GREY = "#949494"        # Benchmark S&P 500
GREY_DARK = "#5c5c5c"   # Benchmark SPXEW

color_map = {
    "value_score": BLUE,
    "momentum_score": ORANGE,
    "passive_GSPC": GREY,
    "passive_SPXEW": GREY_DARK,
}
label_map = {
    "value_score": "Value",
    "momentum_score": "Momentum",
    "passive_GSPC": "Benchmark (S&P 500)",
    "passive_SPXEW": "Benchmark (SPXEW)",
}
order = ["Value", "Momentum", "Benchmark (S&P 500)", "Benchmark (SPXEW)"]
FONT_FAMILY = "Latin Modern Roman, Times New Roman, serif"

df_filtered = df_unified[df_unified["horizon"] == 1].copy()
df_filtered["strategy_label"] = df_filtered["strategy"].map(label_map)

fig = px.line(
    df_filtered,
    x="buyyear",
    y="rendite",
    color="strategy_label",
    color_discrete_map={label_map[k]: v for k, v in color_map.items()},
    category_orders={"strategy_label": order},
    markers=True,
)

fig.update_traces(
    line_width=2.5,
    marker_size=6,
    marker_line_width=0.5,
    marker_line_color="white",
)

fig.update_xaxes(
    title="Kaufjahr",
    dtick=1,
    showline=True,
    linecolor="#333333",
    ticks="outside",
    gridcolor="#e6e6e6",
)

fig.update_yaxes(
    title="Rendite p.a. (%)",
    showline=True,
    linecolor="#333333",
    ticks="outside",
    gridcolor="#e6e6e6",
    zeroline=True,
    zerolinecolor="#333333",
    zerolinewidth=1.2,
)

fig.update_layout(
    title="Rendite nach Kaufjahr (1-Jahres-Horizont)",
    title_x=0.5,
    title_font=dict(size=17, family=FONT_FAMILY),
    font=dict(size=14, family=FONT_FAMILY, color="#222222"),
    width=950,
    height=550,
    template="simple_white",
    margin=dict(l=60, r=40, t=80, b=60),
    legend_title_text=None,
    legend=dict(
        orientation="h",
        yanchor="bottom", y=1.02,
        xanchor="center", x=0.5,
    ),
    plot_bgcolor="white",
    paper_bgcolor="white",
)

fig.show()

output_dir = Path(save_path) / "strategy_returns"
output_dir.mkdir(parents=True, exist_ok=True)
fig.write_image(
    output_dir / "strategy_returns_over_time.png",
    scale=3,
)

In [6]:
df_delta_time = df_unified[df_unified["horizon"] == 1].pivot(
    index="buyyear", columns="strategy", values="rendite"
)
df_delta_time["delta_value"] = df_delta_time["value_score"] - df_delta_time["passive_SPXEW"]
df_delta_time["delta_momentum"] = df_delta_time["momentum_score"] - df_delta_time["passive_SPXEW"]
df_delta_time = df_delta_time.reset_index()

df_plot = df_delta_time.melt(
    id_vars="buyyear",
    value_vars=["delta_value", "delta_momentum"],
    var_name="strategy_delta",
    value_name="delta",
)
delta_label_map = {"delta_value": "Value", "delta_momentum": "Momentum"}
df_plot["strategy_label"] = df_plot["strategy_delta"].map(delta_label_map)

fig = px.line(
    df_plot,
    x="buyyear",
    y="delta",
    color="strategy_label",
    color_discrete_map={"Value": BLUE, "Momentum": ORANGE},
    category_orders={"strategy_label": ["Value", "Momentum"]},
    markers=True,
)

fig.update_traces(
    line_width=2.5,
    marker_size=6,
    marker_line_width=0.5,
    marker_line_color="white",
)

fig.add_hline(y=0, line_width=1.5, line_color="#333333")

fig.add_vrect(
    x0=df_plot["buyyear"].min() - 0.5, x1=2020,
    fillcolor="#e6e6e6", opacity=0.4,
    layer="below", line_width=0,
    annotation_text="eingeschränkte Fundamental-Daten",
    annotation_position="top left",
    annotation_font=dict(size=11, family=FONT_FAMILY, color="#666666"),
)

fig.update_xaxes(
    title="Kaufjahr", dtick=1,
    showline=True, linecolor="#333333", ticks="outside", gridcolor="#e6e6e6",
)
fig.update_yaxes(
    title="Rendite-Delta ggü. SPXEW (Prozentpunkte)",
    showline=True, linecolor="#333333", ticks="outside", gridcolor="#e6e6e6",
    zeroline=False,
)
fig.update_layout(
    title="Outperformance ggü. SPXEW Benchmark über Zeit (1-Jahres-Horizont)",
    title_x=0.5,
    title_font=dict(size=17, family=FONT_FAMILY),
    font=dict(size=14, family=FONT_FAMILY, color="#222222"),
    width=950, height=550,
    template="simple_white",
    margin=dict(l=60, r=40, t=80, b=60),
    legend_title_text=None,
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="center", x=0.5),
    plot_bgcolor="white", paper_bgcolor="white",
)

fig.show()

output_dir = Path(save_path) / "strategy_returns"
output_dir.mkdir(parents=True, exist_ok=True)
fig.write_image(output_dir / "delta_over_time_spxew.png", scale=3)